# Killer migration

Compare one killer motility parameter at a time: polarity decay $a$, polarity
noise $\sigma_\mu$, or translational noise $\sigma_x$.

$$d\boldsymbol{\mu}=-a\boldsymbol{\mu}\,dt+\sigma_\mu\,d\mathbf W_\mu,$$
$$d\mathbf x=v\boldsymbol{\mu}\,dt+\mathbf F\,dt+\sigma_x\,d\mathbf W_x.$$

Smaller $a$ retains polarity longer; $\sigma_\mu$ changes the polarity vector,
whereas $\sigma_x$ adds displacement noise. The drift multiplier $v$ stays fixed,
but changing $a$ also changes polarity magnitude and realized speed. There is no
separate stochastic speed variable.

Mechanics, target motility and cytotoxicity settings stay fixed; contacts and
killing can still respond to migration. These illustrative settings use model
units and are not fitted to experimental trajectories.

In [ ]:
from dataclasses import replace
from pathlib import Path
from time import perf_counter
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.ticker import MaxNLocator
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/mantishrimp/simulation.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the MantiShrimp checkout.")
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import mantishrimp
from mantishrimp import (
    CytotoxicityConfig, DomainConfig, MechanicsConfig, MotilityConfig,
    OutputConfig, SimulationConfig, simulate,
)
if Path(mantishrimp.__file__).resolve() != SRC / "mantishrimp/__init__.py":
    raise RuntimeError("Restart the kernel to load this checkout's simulator.")

import scienceplots
plt.style.use(["science", "no-latex"])
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix", "text.usetex": False,
    # "font.size": 13, "axes.labelsize": 14, "axes.titlesize": 15,
    # "figure.titlesize": 16, "figure.labelsize": 14,
    # "xtick.labelsize": 13, "ytick.labelsize": 13,
    # "xtick.major.size": 7, "ytick.major.size": 7,
    # "xtick.major.width": 1.25, "ytick.major.width": 1.25,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "xtick.top": False, "ytick.right": False,
    "axes.linewidth": 1.1, "axes.spines.top": False, "axes.spines.right": False,
    # "legend.frameon": False, "svg.fonttype": "none", "figure.dpi": 120,
})
FIGURE_DIR = PROJECT_ROOT / "paper/section_2/migration/figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

### 1. Settings

Edit parameters here. Each sweep lists **lower, baseline, higher** values and
matching colour families. `TRAJECTORY_SEED` selects the replicate shown in the figures.

In [ ]:
SEEDS = range(1)
TRAJECTORY_SEED = SEEDS[0]
OBSERVATION_DT = 0.1

BASE = SimulationConfig(
    n_killers=8, n_targets=32,
    killer_radius=10.0, target_radius=12.0,
    duration=20.0, max_dt=0.04, max_drift_displacement=1.0,
    initial_separation_factor=1.0,
    domain=DomainConfig(-300.0, 300.0, -300.0, 300.0, "periodic"),
    motility=MotilityConfig(
        killer_polarity_decay=2.5, killer_polarity_noise=0.75,
        killer_translation_noise=5.0, killer_speed=150.0,
        target_polarity_decay=2.5, target_polarity_noise=0.2,
        target_translation_noise=0.2, target_speed=50.0,
    ),
    mechanics=MechanicsConfig(
        repulsion_kk=250.0, repulsion_tt=250.0, repulsion_kt=250.0,
        adhesion_kt=25.0, binding_rate=3.0, unbinding_rate=0.5,
        capture_radius_factor=1.5, contact_tolerance_factor=0.05,
    ),
    cytotoxicity=CytotoxicityConfig(),
    output=OutputConfig(record_every=1),
)

SWEEPS = [
    ("killer_polarity_decay", "Polarity decay", r"a",
        (0.5, BASE.motility.killer_polarity_decay, 5.0), ("Blue", "Grey", "Teal")),
    ("killer_polarity_noise", "Polarity noise", r"\sigma_\mu",
        (0.2, BASE.motility.killer_polarity_noise, 1.0), ("Purple", "Grey", "Red")),
    ("killer_translation_noise", "Translational noise", r"\sigma_x",
        (0.0, BASE.motility.killer_translation_noise, 50.0), ("Yellow", "Grey", "Orange")),
]

### 2. Conditions and colours

The shared baseline is simulated once per seed. Colours follow the supplied
[Nature palette](https://www.nature.com/documents/natrev-artworkguide.pdf) screenshot,
converted from Apple RGB to sRGB; the sampled values are approximate.

In [ ]:
NATURE_PALETTES = {
    "Grey":   ("#E2E2EA", "#C1C7D5", "#8E99AB", "#647086", "#3F4C62", "#212C3E"),
    "Blue":   ("#C1E4FA", "#92C4E9", "#4C8FCB", "#0068A9", "#004586", "#002753"),
    "Teal":   ("#C5E3EB", "#8CCCCE", "#42B4B5", "#008F99", "#015D6E", "#003245"),
    "Purple": ("#E8D0E6", "#CDA0CB", "#B271AB", "#9E4589", "#6D2769", "#3F1947"),
    "Red":    ("#F9C9C7", "#EA9A9D", "#D95D5B", "#C23637", "#912321", "#691816"),
    "Yellow": ("#FFEFBB", "#F5D77F", "#E8C047", "#C59527", "#926C18", "#614B14"),
    "Orange": ("#FDDBB5", "#FBB874", "#F48F3D", "#E9630E", "#AE460A", "#792C03"),
}
BASE.validate()
if len(SEEDS) == 0 or TRAJECTORY_SEED not in SEEDS:
    raise ValueError("Choose TRAJECTORY_SEED from a non-empty SEEDS collection.")
conditions = {"baseline": BASE}
condition_families = {"baseline": "Grey"}
condition_labels = {"baseline": "Baseline"}
for field, title, symbol, values, families in SWEEPS:
    if not (len(values) == len(families) == 3
            and values[0] < values[1] < values[2]
            and values[1] == getattr(BASE.motility, field)):
        raise ValueError(f"{field}: use three increasing values with the baseline in the middle.")
    for value, family in zip(values, families):
        if value == values[1]:
            continue
        condition = f"{field}={value:g}"
        conditions[condition] = replace(BASE, motility=replace(BASE.motility, **{field: value}))
        conditions[condition].validate()
        condition_families[condition] = family
        condition_labels[condition] = rf"{title}: ${symbol}={value:g}$"
condition_cmaps = {name: LinearSegmentedColormap.from_list(name, NATURE_PALETTES[family])
                   for name, family in condition_families.items()}
condition_table = pd.DataFrame.from_dict({
    name: {field: getattr(config.motility, field) for field, *_ in SWEEPS}
    for name, config in conditions.items()
}, orient="index").rename_axis("condition")
assert not condition_table.duplicated().any()
assert condition_cmaps.keys() == conditions.keys()
assert condition_table.ne(condition_table.loc["baseline"]).sum(axis=1).le(1).all()

if not np.isfinite(OBSERVATION_DT) or OBSERVATION_DT <= 0:
    raise ValueError("OBSERVATION_DT must be finite and positive.")
n_intervals = round(BASE.duration / OBSERVATION_DT)
if n_intervals < 1 or not np.isclose(n_intervals * OBSERVATION_DT, BASE.duration):
    raise ValueError("BASE.duration must be a positive multiple of OBSERVATION_DT.")
observation_times = np.linspace(0.0, BASE.duration, n_intervals + 1)
print(f"{len(conditions)} conditions × {len(SEEDS)} seeds = {len(conditions) * len(SEEDS)} simulations.")
display(condition_table)

# palette_fig, palette_ax = plt.subplots(figsize=(10, 3.3), constrained_layout=True)
# palette_ax.imshow([[to_rgb(c) for c in NATURE_PALETTES[condition_families[name]]]
#                    for name in conditions], aspect="auto", interpolation="nearest")
# palette_ax.set_yticks(range(len(conditions)), [condition_labels[name] for name in conditions])
# palette_ax.set_xticks([0, 5], ["Light", "Dark"])
# palette_ax.tick_params(which="both", length=0, labelsize=11, pad=8)
# for spine in palette_ax.spines.values():
#     spine.set_visible(False)
# for extension in ("svg", "png"):
#     palette_fig.savefig(FIGURE_DIR / f"condition_palette.{extension}", dpi=300, bbox_inches="tight")
# display(palette_fig)
# plt.close(palette_fig)

### 3. Simulate

Run every condition and seed, checking matched initial states. `runs` keeps
unwrapped trajectories and polarity components at the recorded simulation times,
and MSD at the common observation times. The summary reports final RMS displacement
and mean observed speed, with mean and SD across seeds. SD describes variation
between seeds, not uncertainty in the mean.


In [ ]:
def displacements_from_positions(positions, domain):
    """Unwrap (frames, cells, xy) positions and set each initial position to zero."""
    delta = np.diff(positions, axis=0)
    if domain.boundary == "periodic":
        box = np.array([domain.width, domain.height])
        # shortcut: assumes sub-half-box steps; use unwrapped engine output if this fails.
        delta -= box * np.rint(delta / box)
    return np.concatenate([np.zeros_like(positions[:1]), np.cumsum(delta, axis=0)])

runs = {}
initial_by_seed = {}
summary_rows = []
started = perf_counter()

for condition, config in conditions.items():
    print(f"{condition}: {len(SEEDS)} seeds", flush=True)
    for seed in SEEDS:
        result = simulate(replace(config, seed=int(seed)))
        initial = result.snapshots.loc[result.snapshots.step.eq(0)].reset_index(drop=True)
        # Later kills affect pandas dtype inference, not the initial cell state.
        initial["primary_killer"] = initial.primary_killer.astype("Int64")
        if seed in initial_by_seed:
            pd.testing.assert_frame_equal(initial, initial_by_seed[seed])
        else:
            initial_by_seed[seed] = initial
        killers = result.snapshots.loc[result.snapshots.cell_type.eq("killer")].sort_values(
            ["step", "cell_id"]
        )
        times = killers.drop_duplicates("step").time.to_numpy()
        positions = killers[["x", "y"]].to_numpy().reshape(-1, config.n_killers, 2)
        paths = displacements_from_positions(positions, config.domain)
        assert len(times) == len(paths) and np.all(np.diff(times) > 0)
        assert times[0] == 0 and np.isclose(times[-1], config.duration)
        assert np.isfinite(paths).all()
        sampled = np.empty((len(observation_times), config.n_killers, 2))
        for cell in range(config.n_killers):
            for axis in range(2):
                sampled[:, cell, axis] = np.interp(observation_times, times, paths[:, cell, axis])
        msd = np.mean(np.sum(sampled ** 2, axis=2), axis=1)
        observed_speeds = np.linalg.norm(np.diff(sampled, axis=0), axis=2) / OBSERVATION_DT
        polarity = killers[["polarity_x", "polarity_y"]].to_numpy().reshape(
            -1, config.n_killers, 2
        )
        assert polarity.shape == paths.shape and np.isfinite(polarity).all()
        runs[(condition, seed)] = {
            "times": times, "paths": paths, "msd": msd, "polarity": polarity,
        }
        summary_rows.append({
            "condition": condition, "seed": seed,
            "rms_displacement": np.sqrt(msd[-1]),
            "mean_observed_speed": observed_speeds.mean(),
        })
        del result, killers, positions, sampled, polarity
run_summary = pd.DataFrame(summary_rows)
summary = condition_table.join(run_summary.groupby("condition", sort=False).agg(
    rms_displacement_mean=("rms_displacement", "mean"),
    rms_displacement_sd=("rms_displacement", "std"),
    observed_speed_mean=("mean_observed_speed", "mean"),
    observed_speed_sd=("mean_observed_speed", "std"),
))
print(f"Completed in {perf_counter() - started:.1f} s.")
display(summary.round(3))

### 4. Trajectory comparisons

Each figure compares the lower, baseline and higher values using the same seed.
Trajectories start at zero; **+** marks the start and circles mark endpoints.
Shades distinguish killers within a condition. Axis scales match within each
figure and can differ between figures. SVGs and PNG previews are saved in `figures/`.

In [ ]:
for field, title, symbol, values, _ in SWEEPS:
    names = ["baseline" if value == values[1] else f"{field}={value:g}" for value in values]
    paths_by_condition = [runs[(name, TRAJECTORY_SEED)]["paths"] for name in names]
    extent = max(1.0, max(float(np.abs(paths).max()) for paths in paths_by_condition) * 1.08)
    fig, axes = plt.subplots(1, 3, figsize=(11.6, 4.4), sharex=True, sharey=True,
                                constrained_layout=True)
    for ax, name, paths, value in zip(axes, names, paths_by_condition, values):
        colors = condition_cmaps[name](np.linspace(0.35, 0.9, BASE.n_killers))
        for cell, color in enumerate(colors):
            ax.plot(paths[:, cell, 0], paths[:, cell, 1], color=color, lw=1.4)
        ax.scatter(paths[-1, :, 0], paths[-1, :, 1], c=colors, s=20, zorder=3)
        ax.scatter(0, 0, color="black", marker="+", s=55, linewidth=1.4, zorder=4)
        ax.set(xlim=(-extent, extent), ylim=(-extent, extent), aspect="equal")
        label = rf"${symbol}={value:g}$" + (" (baseline)" if name == "baseline" else "")
        ax.set_title(label, pad=12)
        ax.xaxis.set_major_locator(MaxNLocator(3))
        ax.yaxis.set_major_locator(MaxNLocator(3))
    axes[0].set_ylabel("y displacement (model units)")
    fig.supxlabel("x displacement (model units)")
    fig.suptitle(f"{title} | seed {TRAJECTORY_SEED} | {BASE.n_killers} killers per condition")
    filename = f"trajectory_comparison_{field.removeprefix('killer_')}"
    # for extension in ("svg", "png"):
    #     fig.savefig(FIGURE_DIR / f"{filename}.{extension}", dpi=300, bbox_inches="tight")
    display(fig)
    plt.close(fig)

In [ ]:
panels = [
    ("baseline", "Teal", r"$\alpha = 2.5\,\mathrm{min}^{-1}$"),
    ("killer_polarity_decay=0.5", "Blue", r"$\alpha = 0.5\,\mathrm{min}^{-1}$"),
]
paths_to_plot = [runs[(name, TRAJECTORY_SEED)]["paths"] for name, _, _ in panels]
extent = max(1.0, max(np.abs(paths).max() for paths in paths_to_plot) * 1.08)

fig, axes = plt.subplots(
    1, 2, figsize=(8, 4.4),
    sharex=True, sharey=True, constrained_layout=True,
    dpi = 300
)

for ax, (_, family, title), paths in zip(axes, panels, paths_to_plot):
    cmap = LinearSegmentedColormap.from_list(family, NATURE_PALETTES[family])
    colors = cmap(np.linspace(0.35, 0.9, paths.shape[1]))

    for cell, color in enumerate(colors):
        ax.plot(paths[:, cell, 0], paths[:, cell, 1], color=color, lw=1.4)

    # ax.scatter(paths[-1, :, 0], paths[-1, :, 1], c=colors, s=20, zorder=3)
    # ax.scatter(0, 0, color="black", marker="+", s=55, linewidth=1.4, zorder=4)
    ax.set(xlim=(-extent, extent), ylim=(-extent, extent), aspect="equal")
    ax.set_title(title, fontsize=20, pad=12)
    ax.tick_params(labelsize=16, length=7, width=1.25)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))

axes[0].set_ylabel("y displacement (µm)", fontsize=24, fontstyle="normal")
fig.supxlabel("x displacement (µm)", fontsize=24, fontstyle="normal")

filename = "trajectory_alpha"
for extension in ("svg", "pdf"):
    fig.savefig(
        FIGURE_DIR / f"{filename}.{extension}",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )
display(fig)
plt.close(fig)

In [ ]:
panels = [
    ("killer_polarity_noise=0.2", "Purple", r"$\sigma_\mu = 0.2\,\mathrm{min}^{-1/2}$"),
    ("killer_polarity_noise=1", "Red", r"$\sigma_\mu = 1\,\mathrm{min}^{-1/2}$"),
]

paths_to_plot = [runs[(name, TRAJECTORY_SEED)]["paths"] for name, _, _ in panels]
extent = max(1.0, max(np.abs(paths).max() for paths in paths_to_plot) * 1.08)

fig, axes = plt.subplots(
    1, 2, figsize=(8, 4.4),
    sharex=True, sharey=True,
    constrained_layout=True, dpi=300,
)

for ax, (_, family, title), paths in zip(axes, panels, paths_to_plot):
    cmap = LinearSegmentedColormap.from_list(family, NATURE_PALETTES[family])
    colors = cmap(np.linspace(0.35, 0.9, paths.shape[1]))

    for cell, color in enumerate(colors):
        ax.plot(paths[:, cell, 0], paths[:, cell, 1], color=color, lw=1.4)

    ax.set(xlim=(-extent, extent), ylim=(-extent, extent), aspect="equal")
    ax.set_title(title, fontsize=20, pad=12)
    ax.tick_params(labelsize=16, length=7, width=1.25)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))

axes[0].set_ylabel("y displacement (µm)", fontsize=24, fontstyle="normal")
fig.supxlabel("x displacement (µm)", fontsize=24, fontstyle="normal")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
filename = "trajectory_sigma_mu"

for extension in ("svg", "png", "pdf"):
    fig.savefig(
        FIGURE_DIR / f"{filename}.{extension}",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

display(fig)
plt.close(fig)

# Uses existing runs; no simulations are rerun.

In [ ]:
panels = [
    ("killer_translation_noise=0", "Yellow",
        r"$\sigma_r = $0 µm min$^{-1/2}$"),
    ("killer_translation_noise=50", "Orange",
        r"$\sigma_r = 50$ µm min$^{-1/2}$"),
]

paths_to_plot = [
    runs[(name, TRAJECTORY_SEED)]["paths"]
    for name, _, _ in panels
]
extent = max(
    1.0,
    max(np.abs(paths).max() for paths in paths_to_plot) * 1.08,
)

fig, axes = plt.subplots(
    1, 2, figsize=(8, 4.4),
    sharex=True, sharey=True,
    constrained_layout=True, dpi=300,
)

for ax, (_, family, title), paths in zip(axes, panels, paths_to_plot):
    cmap = LinearSegmentedColormap.from_list(
        family, NATURE_PALETTES[family]
    )
    colors = cmap(np.linspace(0.35, 0.9, paths.shape[1]))

    for cell, color in enumerate(colors):
        ax.plot(
            paths[:, cell, 0], paths[:, cell, 1],
            color=color, lw=1.4,
        )

    ax.set(
        xlim=(-extent, extent),
        ylim=(-extent, extent),
        aspect="equal",
    )
    ax.set_title(title, fontsize=20, pad=12)
    ax.tick_params(labelsize=16, length=7, width=1.25)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.yaxis.set_major_locator(MaxNLocator(3))

axes[0].set_ylabel(
    "y displacement (µm)", fontsize=24, fontstyle="normal"
)
fig.supxlabel(
    "x displacement (µm)", fontsize=24, fontstyle="normal"
)

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
filename = "trajectory_sigma_r"

for extension in ("svg", "pdf"):
    fig.savefig(
        FIGURE_DIR / f"{filename}.{extension}",
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )

display(fig)
plt.close(fig)